# What does cross-validation actually decide?

`RidgeClassifierCV` tries three settings for `alpha` and keeps one.
sklearn tells you which. It does not tell you why. Let's make the code
itself explain.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
from sklearn.linear_model import RidgeClassifierCV

def fit_coef(X, yraw):
    y = (yraw > 0.0).astype(float)
    return RidgeClassifierCV(alphas=[0.1, 1.0, 10.0]).fit(X, y).coef_.ravel()

rng = np.random.default_rng(0)
X, yraw = rng.standard_normal((8, 2)), rng.standard_normal(8)

clf = RidgeClassifierCV(alphas=[0.1, 1.0, 10.0]).fit(X, (yraw > 0).astype(float))
print("coefficients:", np.round(clf.coef_.ravel(), 6))
print("alpha picked:", clf.alpha_)

coefficients: [0.037205 0.153496]
alpha picked: 10.0


Now trace the same fit. `verify` runs it once, checks its answer
against sklearn's, and hands back the mathematics it did:

In [2]:
from skverify import latex
from skverify import to_sympy

out = to_sympy(fit_coef, X, yraw)
print("traced coefficients:", np.round(np.asarray(out.value, float).ravel(), 6))

traced coefficients: [0.037205 0.153496]


Same numbers. Here is the reason the model picked alpha = 10,
written down by the trace as two conditions:

In [3]:
from IPython.display import Math, display
legend = {}
for g in out.preconditions.args:
    if "solve_eigen" in str(g):
        display(Math(latex(g, aliases=legend)))
print("T1, T2, T3 = the mistakes each alpha made (0.1, 1.0, 10.0), one per data point")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

T1, T2, T3 = the mistakes each alpha made (0.1, 1.0, 10.0), one per data point


Read them like sentences: alpha 1.0 made smaller mistakes than
alpha 0.1, and alpha 10.0 made smaller mistakes than alpha 0.1 too.
The best of the survivors wins. sklearn's own numbers agree:

In [4]:
clf = RidgeClassifierCV(alphas=[0.1, 1.0, 10.0], store_cv_results=True).fit(
    X, (yraw > 0).astype(float))
errors = ((clf.cv_results_[:, 0, :] - np.where(yraw > 0, 1.0, -1.0)[:, None]) ** 2).mean(0)
for a, e in zip([0.1, 1.0, 10.0], errors):
    print(f"alpha {a:>4}: average squared mistake {e:7.3f}" + ("   <- picked" if a == clf.alpha_ else ""))

alpha  0.1: average squared mistake  22.451
alpha  1.0: average squared mistake   8.028
alpha 10.0: average squared mistake   3.318   <- picked


One more thing the conditions give us for free: they say exactly
where the model's behavior changes. `yraw[0] > 0` is one of them, so
put that sample exactly on the boundary and watch:

In [5]:
for v, story in [(0.5, "clearly class 1"), (0.0, "exactly on the boundary")]:
    y2 = yraw.copy(); y2[0] = v
    print(f"yraw[0] = {v}  ({story:23s}) -> coefficients {np.round(fit_coef(X, y2), 4)}")

yraw[0] = 0.5  (clearly class 1        ) -> coefficients [0.0979 0.1365]
yraw[0] = 0.0  (exactly on the boundary) -> coefficients [0.0372 0.1535]


The coefficients jump: at exactly 0 the sample switches class, and
the model refits. The trace named that boundary before we tried it.

**In short:** the traced fit matches sklearn to the last digit, the
alpha choice is two one-line conditions you can check, and every
condition marks a boundary you can poke. Nobody read sklearn's source
to learn any of this.